# 5주차 과제 베이스라인 — 항공편 지연 예측

미국 항공편 정보로 지연 여부(`Delayed`/`Not_Delayed`)를 예측합니다. 원본 `train.csv`는 100만 행이며, 그중 **74.5%는 정답(`Delay`)이 없는 행**입니다. 실제 데이터에서도 라벨을 확인할 수 없는 행이 포함될 수 있습니다. 이 베이스라인은 정답이 있는 행을 실습 크기로 줄이고, **분할 뒤 학습 데이터에만 맞추는 전처리 Pipeline과 로지스틱 회귀 한 개**를 바로 실행할 수 있게 제공합니다. 성공한 점수보다 실행·오류·문제 분해와 다음 행동을 남기는 일이 더 중요합니다. 원본 파일이 없으면 두 클래스를 포함한 **연습용 소규모 예시 데이터**로 전환되며, 그 지표는 실제 항공편에 대한 결론이 아닙니다.


## 1. 데이터 불러오기 & 샘플링

원본 데이터와 변수 설명은 [데이콘 항공편 지연 예측 경진대회](https://dacon.io/competitions/official/236094) 페이지에서 확인하고 내려받을 수 있습니다. 페이지에 로그인한 뒤 데이터 탭의 안내에 따라 내려받습니다. 압축을 푼 파일을 아래 코드에 적힌 경로에 둡니다. 다음 셀은 파일 상태를 보여 주고, 원본이 없으면 연습용 데이터로 실행 흐름을 이어 갑니다.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

train_path = Path('../dataset/open/extracted/월간 데이콘 항공편 지연 예측 AI 경진대회/train.csv')
print('현재 작업 폴더:', Path.cwd())
numeric_features = ['Estimated_Departure_Time', 'Estimated_Arrival_Time']
categorical_features = [
    'Origin_State', 'Destination_State',
    'Airline', 'Carrier_Code(IATA)',
]
usecols = ['ID', 'Tail_Number', 'Delay'] + numeric_features + categorical_features
print('train.csv 존재:', train_path.exists())

if train_path.exists():
    DATA_MODE = '실제 데이터'
    raw = pd.read_csv(train_path, usecols=usecols)
else:
    DATA_MODE = '연습용 소규모 예시 데이터'
    states = ['CA', 'NY', 'TX', 'WA']
    airlines = ['Demo Air', 'Sample Jet', 'Practice Line']
    demo_rows = []
    for i in range(120):
        delayed = i % 5 == 0
        departure = 500 + (i % 18) * 100
        demo_rows.append({
            'ID': f'DEMO_{i:03d}',
            'Tail_Number': f'DEMO_TAIL_{i % 12:02d}',
            'Delay': 'Delayed' if delayed else 'Not_Delayed',
            'Estimated_Departure_Time': departure,
            'Estimated_Arrival_Time': departure + 130 + (40 if delayed else 0),
            'Origin_State': states[i % len(states)],
            'Destination_State': states[(i * 3 + 1) % len(states)],
            'Airline': airlines[i % len(airlines)],
            'Carrier_Code(IATA)': f'D{i % len(airlines)}',
        })
    raw = pd.DataFrame(demo_rows, columns=usecols)
    print('train.csv가 없어 예시 데이터로 코드 흐름만 연습합니다.')
    print('예시 분류 지표는 실제 항공편 지연 성능이 아닙니다.')

print('데이터 모드:', DATA_MODE)
print('필요한 열만 읽은 크기:', raw.shape)


In [ ]:
# 정답이 있는 행만 사용
labeled = raw.dropna(subset=['Delay']).reset_index(drop=True)
print('정답 있는 행:', labeled.shape[0], '/ 전체:', raw.shape[0])
labeled['Delay'].value_counts(normalize=True)


실제 원본의 지연 비율은 약 17.6%로 뚜렷한 **불균형 데이터**입니다. 연습용 예시 데이터는 안전한 분할을 위해 두 클래스를 충분히 넣어 만든 별도 분포이므로 이 수치와 비교하지 않습니다. 실제 데이터가 크면 `train_test_split`의 `stratify`로 클래스 비율을 유지하면서 2만 행으로 줄이고, 예시 데이터 모드에서는 모든 행을 사용합니다.


In [ ]:
SAMPLE_SIZE = 20000  # 실제 데이터에서 메모리가 부족하면 5000으로 낮추어도 됩니다.
if len(labeled) > SAMPLE_SIZE:
    sample, _ = train_test_split(
        labeled, train_size=SAMPLE_SIZE, stratify=labeled['Delay'], random_state=42
    )
    sample = sample.reset_index(drop=True)
else:
    sample = labeled.sample(frac=1, random_state=42).reset_index(drop=True)
    print('전체 정답 행이 샘플 크기보다 작아 모든 행을 사용합니다.')
print(sample.shape)
sample['Delay'].value_counts(normalize=True)


## 2. 식별자 제거와 결측치 확인


In [ ]:
sample.isna().sum()


In [ ]:
# 식별자 성격의 컬럼은 이번 기본 시도의 예측 변수에서 제외합니다.
sample = sample.drop(columns=['ID', 'Tail_Number'])

sample[numeric_features + categorical_features].isna().sum()


In [ ]:
sample.head()


## 이번 주 과제 경로

- **✅ 기본 시도**: 아래 셀을 순서대로 실행해 로지스틱 회귀 한 개와 지연 클래스 지표를 확인합니다.
- **🧩 문제 분해**: 멈추면 마지막 성공 셀, 오류 마지막 줄, 의심 원인 하나, 작은 확인 코드 하나를 기록합니다. 해결하지 못해도 괜찮습니다.
- **🌱 선택 탐색**: 여유가 있을 때만 더미 기준선, 임계값, 시간 분할, 다른 모델, PR 곡선 중 하나를 고릅니다.

기본 완료 기준은 **질문 1개 + 실행 또는 실행 시도 1개 + 관찰 결과 또는 오류 1개 + 다음 행동 1개**입니다.


### 3. ✅ 기본 시도 — 원본 입력과 타깃 이진화

`Delay`를 0/1로 바꾸고, 먼저 여섯 개 변수로 시작합니다. 결측치 대체와 범주형 인코딩은 아직 수행하지 않습니다. 더 많은 변수를 사용하고 싶다면 범주 수와 메모리 사용량을 확인한 뒤 선택 탐색으로 추가합니다.


In [ ]:
X = sample[numeric_features + categorical_features].copy()
y = (sample['Delay'] == 'Delayed').astype(int)
print(X.shape, y.mean().round(3))


### 4. ✅ 기본 시도 — 학습/검증 데이터 분리

`stratify=y`는 두 세트의 지연 비율을 비슷하게 유지합니다. 이는 누수 방지나 시간 대표성까지 보장하는 옵션은 아닙니다.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('학습 / 검증 크기:', X_train.shape, X_valid.shape)
print('지연 비율:', y_train.mean().round(3), y_valid.mean().round(3))


### 5. ✅ 기본 시도 — 로지스틱 회귀 한 개 학습

Pipeline 전체를 `X_train`에만 fit하므로 중앙값·최빈값·범주 목록도 학습 데이터에서만 정해집니다.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_features),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_features),
])

def make_pipeline(model):
    return Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', model),
    ])

model = make_pipeline(LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
pred_valid = model.predict(X_valid)
print('학습과 예측을 마쳤습니다.')


### 6. ✅ 기본 시도 — 지연 클래스 지표 읽기

코드는 여러 지표를 한 번에 보여 주지만, 처음에는 **지연 recall 또는 F1 하나만** 골라 자신의 말로 해석해도 됩니다. 지연을 양성(1)으로 두었다는 점을 함께 적습니다.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

scores = {
    'accuracy': accuracy_score(y_valid, pred_valid),
    '지연 precision': precision_score(y_valid, pred_valid, zero_division=0),
    '지연 recall': recall_score(y_valid, pred_valid, zero_division=0),
    '지연 F1': f1_score(y_valid, pred_valid, zero_division=0),
}
print('평가 데이터 모드:', DATA_MODE)
pd.Series(scores).round(3)


### 7. 🌱 선택 탐색 — 불균형 기준선 또는 다른 질문 하나

아래 셀은 기본 범위에서 실행하지 않아도 됩니다. `RUN_OPTIONAL=True`로 바꾸면 항상 다수 클래스만 예측하는 기준선과 비교합니다. 대신 임계값, 시간 분할, 다른 모델, PR 곡선 중 하나를 탐색해도 됩니다.


In [ ]:
RUN_OPTIONAL = False

if RUN_OPTIONAL:
    from sklearn.dummy import DummyClassifier
    dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
    dummy_pred = dummy.predict(X_valid)
    print('더미 accuracy:', round(accuracy_score(y_valid, dummy_pred), 3))
    print('더미 지연 recall:', round(recall_score(y_valid, dummy_pred, zero_division=0), 3))
else:
    print('선택 탐색을 건너뛰었습니다. 기본 시도 기록으로 이동합니다.')


## 여기까지 하면 이번 주 기록 완료

아래 네 줄을 이 셀 아래 새 Markdown 셀에 적습니다. 모델이 실행되지 않았어도 오류를 나누어 보고 다음 행동을 정했다면 완료입니다.

- 질문 1개:
- 실행 또는 실행 시도 1개:
- 관찰한 결과 또는 오류 1개:
- 다음 행동 1개:
